#  Machine Learning — v11 (Multi-Output sobre 168_ytva_feature_extraction2)

Predicción simultánea de **6 targets categóricos** a partir de **~73 features numéricas** del bloque entrainment.

| | |
|---|---|
| **Features (X)** | columnas `HP_n_peaks` → `COS_peak_phase_std` (entrainment, sin FR_/MC_) |
| **Targets (Y)** | `FRc_wv_period_w`, `FRc_n_peaks`, `FRc_autocorr_val`, `MCc_rhythmic`, `FRc_mean_peak_value_norm`, `FRc_std_peak_value_relativ` |
| **Aproximación** | `MultiOutputClassifier` de sklearn — UN objeto que predice los 6 a la vez |
| **Modelos comparados** | 10 (AdaBoost, RandomForest, GradBoost, ExtraTrees, SVM, KNN, LogReg, MLP, XGBoost, LightGBM) |

**Importante (anti-leakage)**: las features se limitan al bloque entrainment (`HP_`, `LP_`, `RAW_T_`, `WV_`, `D1/2/3_`, `COS_`). Los bloques `FR_`/`MC_` están excluidos porque las `FRc_*`/`MCc_*` (targets) se derivan literalmente de ellos.

In [ ]:
pip install shap

## Bloque 1 — Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier      # ⭐ multi-output
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc
)

# ─── Clasificadores ─────────────────────────────────────────────────────────
from sklearn.ensemble import (
    AdaBoostClassifier, RandomForestClassifier,
    GradientBoostingClassifier, ExtraTreesClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_OK = True
except ImportError:
    XGBOOST_OK = False
    print("⚠️  XGBoost no instalado")

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_OK = True
except ImportError:
    LIGHTGBM_OK = False
    print("⚠️  LightGBM no instalado")
# ⭐ NUEVO: feature selection + CV + SHAP
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import KFold

try:
    import shap
    SHAP_OK = True
    print(f'✅ SHAP versión: {shap.__version__}')
except ImportError:
    SHAP_OK = False
    print('⚠️  SHAP no instalado. Ejecuta:  pip install shap')

import warnings
warnings.filterwarnings('ignore', category=UserWarning,
                        message='X does not have valid feature names')

## Bloque 2 — Configuración visual (copiada de tu v8)

In [ ]:
# ── Paleta y colores ──────────────────────────────────────────────────────────
PALETTE    = ["#264653", "#2a9d8f", "#e9c46a", "#f4a261", "#e76f51",
              "#457b9d", "#a8dadc", "#e63946", "#606c38", "#dda15e"]
CMAP_CM    = "Blues"
CMAP_FIMP  = "YlOrRd"
BG_COLOR   = "#FFFFFF"
ACCENT     = "#2a9d8f"

# ── Tamaños POSTER ────────────────────────────────────────────────────────────
FONT_TITLE  = 18
FONT_LABEL  = 15
FONT_TICK   = 13
FONT_LEGEND = 13
FONT_ANNOT  = 12

plt.rcParams.update({
    "figure.facecolor":  BG_COLOR,
    "axes.facecolor":    BG_COLOR,
    "font.family":       "DejaVu Sans",
    "font.size":         FONT_TICK,
    "axes.titlesize":    FONT_TITLE,
    "axes.labelsize":    FONT_LABEL,
    "xtick.labelsize":   FONT_TICK,
    "ytick.labelsize":   FONT_TICK,
    "legend.fontsize":   FONT_LEGEND,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

LANG    = "ENG"
carpeta = f"Figuras_v11_multioutput_6_{LANG}"
sufijo  = f"_{LANG}"
#os.makedirs(carpeta, exist_ok=True)
#print(f"✓ Carpeta lista: /{carpeta}/")

## Bloque 3 — Cargar datos + definir features y targets

**Selección robusta del rango de features**: en vez de usar índices numéricos (`iloc[:, 7:80]`),
se define el rango por **nombre de columna**. Si mañana añades/quitas columnas, sigue funcionando.

In [ ]:
# ─── Cargar dbm ──────────────────────────────────────────────────────────────
df = pd.read_excel('168_ytva_feature_extraction2.xlsx')
print(f"Shape: {df.shape}")
print(f"Columnas (primeras 10): {df.columns[:10].tolist()}")
print(f"Columnas (últimas 10):  {df.columns[-10:].tolist()}")
df.head(2)

In [ ]:
# ─── Definir TARGETS y FEAT_COLS ─────────────────────────────────────────────
TARGETS = [
    'FRc_wv_period_w',
    'FRc_n_peaks',
    'FRc_autocorr_val',
    'MCc_rhythmic',
    'FRc_mean_peak_value_norm',
    'FRc_std_peak_value_relativ',
]

# Rango de features: HP_n_peaks → COS_peak_phase_std (entrainment)
FEAT_START_COL = 'HP_n_peaks'
FEAT_END_COL   = 'COS_peak_phase_std'

cols      = df.columns.tolist()
i_start   = cols.index(FEAT_START_COL)
i_end     = cols.index(FEAT_END_COL)
feat_raw  = cols[i_start : i_end + 1]

# Filtrar a numéricas (descarta cualquier cosa no numérica que se cuele)
FEAT_COLS = df[feat_raw].select_dtypes(include='number').columns.tolist()

print(f"✅ Features ({len(FEAT_COLS)}): {FEAT_START_COL} → {FEAT_END_COL}")
print(f"✅ Targets  ({len(TARGETS)}):")
for t in TARGETS:
    n_classes = df[t].nunique()
    print(f"     - {t:<35} → {n_classes} clases: {sorted(df[t].dropna().unique().tolist())}")

In [ ]:
# --- Fix de compatibilidad pandas >=3.0 ---
# En pandas 2.x, Y_raw = df[TARGETS].astype(str) convertia los NaN en
# la cadena "nan", que LabelEncoder trataba como una clase de texto mas.
# Desde pandas 3.0, .astype(str) preserva los NaN como float en vez de
# convertirlos a texto, lo que rompe mas adelante classification_report
# (una de las "clases" deja de ser str). Se elimina aqui la unica fila
# (de 150) sin etiqueta en alguno de los 6 targets, antes de construir X/Y.
n_before = len(df)
df = df.dropna(subset=TARGETS).reset_index(drop=True)
print(f"Filas eliminadas por NaN en TARGETS: {n_before - len(df)} (de {n_before})")


## Bloque 4 — Encodar targets y preparar X / Y

- **X**: matriz `(n_samples, n_features)` numérica, NaN imputados con la media.
- **Y**: matriz `(n_samples, 6)` con cada target encodado a entero por un `LabelEncoder` distinto.
- **Split**: `train_test_split` sin `stratify` (sklearn no soporta stratify sobre Y multi-output sin librería externa). `random_state=42` para reproducibilidad.

In [ ]:
# ─── Diagnóstico: dónde están los NaN ──────────────────────────────────────── DECISION= QUITAR COLUMNAS D PARÁMETROS CON VALORES NaN en vez d las MUESTRAS EN SÍ
X_check = df[FEAT_COLS].copy().to_numpy()

# Por columna
n_nan_por_col = np.isnan(X_check).sum(axis=0)
pct_nan_col   = 100 * n_nan_por_col / X_check.shape[0]

print(f"📊 Distribución de NaN (n_filas total = {X_check.shape[0]}):\n")
print(f"Columnas con NaN ordenadas (top 15):")
for i in np.argsort(n_nan_por_col)[::-1][:15]:
    if n_nan_por_col[i] > 0:
        print(f"  {FEAT_COLS[i]:<35} {n_nan_por_col[i]:>4} NaN ({pct_nan_col[i]:5.1f}%)")

# Cuántas filas tienen NaN si quitamos las columnas con >X% NaN
for umbral in [50, 30, 20, 10, 5]:
    cols_ok    = [i for i, p in enumerate(pct_nan_col) if p <= umbral]
    X_red      = X_check[:, cols_ok]
    n_filas_ok = (~np.isnan(X_red).any(axis=1)).sum()
    print(f"\nSi quitas columnas con >{umbral}% NaN: "
          f"quedan {len(cols_ok)} features y {n_filas_ok} filas sin NaN")

In [ ]:
# ─── Filtrar columnas con demasiados NaN ─────────────────────────────────────
UMBRAL_NAN_COL = 10   # ← columnas con más de este % de NaN se eliminan

X_check       = df[FEAT_COLS].copy().to_numpy()
pct_nan_col   = 100 * np.isnan(X_check).sum(axis=0) / X_check.shape[0]
cols_dropped  = [c for c, p in zip(FEAT_COLS, pct_nan_col) if p > UMBRAL_NAN_COL]
FEAT_COLS_OK  = [c for c, p in zip(FEAT_COLS, pct_nan_col) if p <= UMBRAL_NAN_COL]

print(f"Features eliminadas (>{UMBRAL_NAN_COL}% NaN): {cols_dropped}")
print(f"Features que quedan: {len(FEAT_COLS_OK)} (de {len(FEAT_COLS)} originales)")

# ─── Preparar X con las columnas filtradas ───────────────────────────────────
X = df[FEAT_COLS_OK].copy().to_numpy()

# ─── Preparar Y (encoder por target) ─────────────────────────────────────────
Y_raw    = df[TARGETS].astype(str)            # forzar str (StringArray → str)
encoders = {t: LabelEncoder().fit(Y_raw[t]) for t in TARGETS}
Y        = np.column_stack([encoders[t].transform(Y_raw[t]) for t in TARGETS])

# ─── Safety net: filtrar las pocas filas que aún tengan NaN ─────────────────
# (con UMBRAL=10 en tu caso no debería quitar ninguna, pero por si las moscas)
mask_no_nan = ~np.isnan(X).any(axis=1)
X = X[mask_no_nan]
Y = Y[mask_no_nan]

# ⭐ Actualizar FEAT_COLS globalmente — necesario para Bloques 13 y 14 (feature importance)
FEAT_COLS = FEAT_COLS_OK

print(f"\n✅ X shape: {X.shape}")
print(f"✅ Y shape: {Y.shape}  (cada columna = 1 target encodado a int)")
print()
for t in TARGETS:
    print(f"  {t:<35} → clases: {list(encoders[t].classes_)}")

In [ ]:
# ─── Train / test split ──────────────────────────────────────────────────────
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)
print(f"X_train: {X_train.shape}  Y_train: {Y_train.shape}")
print(f"X_test : {X_test.shape}  Y_test : {Y_test.shape}")

## Bloque 5 — Funciones de entrenamiento Multi-Output

El truco: cada clasificador se envuelve en `MultiOutputClassifier(base)`.
Internamente sklearn entrena un estimador por target (acceso vía `.estimators_[i]`) — esto es exactamente lo mismo que haría tu código v8 con un bucle, pero ahora es **un solo objeto** que predice los 6 a la vez con `.predict(X)`.

Para feature importance se usa la misma lógica que tu `entrenar_modelo` de v8:
- `feature_importances_` nativo (árboles, boosting)
- `|coef_|` promediado sobre clases (lineales)
- `permutation_importance` (SVM-RBF, KNN, MLP)

In [ ]:
# ─── FACTORY: crea instancia nueva de un clasificador por nombre ────────────
def _build_modelo(nombre):
    if nombre == "AdaBoost":     return AdaBoostClassifier(n_estimators=100, random_state=42)
    if nombre == "RandomForest": return RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
    if nombre == "GradBoost":    return GradientBoostingClassifier(n_estimators=100, random_state=42)
    if nombre == "ExtraTrees":   return ExtraTreesClassifier(n_estimators=100, random_state=42)
    if nombre == "SVM":          return SVC(random_state=42, probability=True)
    if nombre == "KNN":          return KNeighborsClassifier(n_neighbors=5)
    if nombre == "LogReg":       return LogisticRegression(max_iter=1000, random_state=42)
    if nombre == "MLP":          return MLPClassifier(max_iter=500, random_state=42)
    if nombre == "XGBoost":
        if not XGBOOST_OK: raise ImportError("xgboost no instalado")
        return XGBClassifier(n_estimators=100, random_state=42, verbosity=0)
    if nombre == "LightGBM":
        if not LIGHTGBM_OK: raise ImportError("lightgbm no instalado")
        return LGBMClassifier(n_estimators=100, random_state=42, class_weight="balanced", verbose=-1)
    raise ValueError(f"Modelo desconocido: {nombre}")


# ─── HELPER: extrae feature_importance de UN sub-estimador ──────────────────
def _feat_imp_one(est, X_test, y_test, feat_names):
    """Devuelve pd.Series de feature importance para UN estimador entrenado."""
    if hasattr(est, "feature_importances_"):
        return pd.Series(est.feature_importances_, index=feat_names)
    if hasattr(est, "coef_") and est.coef_.ndim >= 1:
        coef = est.coef_
        vals = np.abs(coef) if coef.ndim == 1 else np.abs(coef).mean(axis=0)
        return pd.Series(vals, index=feat_names)
    # Fallback: permutation importance
    pi = permutation_importance(est, X_test, y_test, n_repeats=5,
                                 random_state=42, n_jobs=-1)
    return pd.Series(pi.importances_mean, index=feat_names)


# ─── ENTRENA UN MultiOutputClassifier + extrae todo ─────────────────────────
def entrenar_multioutput(X_train, X_test, Y_train, Y_test, nombre_modelo,
                          targets, encoders, feat_names):
    """
    Entrena MultiOutputClassifier(<base>) y devuelve dict con:
      - modelo            : el MultiOutputClassifier entrenado
      - Y_pred            : predicciones (n_test, n_targets) encodadas a int
      - Y_proba           : LISTA de arrays (uno por target) con probabilidades
      - acc_per_target    : dict {target: accuracy}
      - acc_mean          : media de las accuracies por target
      - acc_subset        : % de filas donde TODOS los targets son correctos
      - feat_imp_per_target : dict {target: Series}
      - feat_imp_mean     : Series (media de importance sobre los 6 targets)
    """
    base   = _build_modelo(nombre_modelo)
    modelo = MultiOutputClassifier(base, n_jobs=-1)
    modelo.fit(X_train, Y_train)

    Y_pred  = modelo.predict(X_test)
    Y_proba = modelo.predict_proba(X_test)   # ← lista de 6 arrays

    # Accuracy por target
    acc_per = {t: accuracy_score(Y_test[:, i], Y_pred[:, i])
               for i, t in enumerate(targets)}
    acc_mean   = float(np.mean(list(acc_per.values())))
    # ⭐ accuracy_score de sklearn no soporta multi-class multi-output;
    #    calculamos subset acc (= % filas con TODOS los targets correctos) a mano.
    acc_subset = float(np.all(Y_pred == Y_test, axis=1).mean())

    # Feature importance por target
    feat_imp_per = {}
    for i, t in enumerate(targets):
        feat_imp_per[t] = _feat_imp_one(
            modelo.estimators_[i], X_test, Y_test[:, i], feat_names
        )
    feat_imp_mean = pd.concat(feat_imp_per.values(), axis=1).mean(axis=1)

    return dict(
        modelo=modelo, Y_pred=Y_pred, Y_proba=Y_proba,
        Y_test=Y_test, X_test=X_test,
        acc_per_target=acc_per, acc_mean=acc_mean, acc_subset=acc_subset,
        feat_imp_per_target=feat_imp_per, feat_imp_mean=feat_imp_mean,
        nombre_modelo=nombre_modelo,
    )

print("✓ Funciones listas")

## Bloque 6 — Entrenar TODOS los modelos

Para cada uno de los 10 clasificadores: `MultiOutputClassifier(base)` → fit → predict → métricas.
El resultado es `todos_res[nombre_modelo] = dict(...)` con accuracy por target, subset accuracy, etc.

In [ ]:
nombres_modelos = ["AdaBoost", "RandomForest", "GradBoost", "ExtraTrees",
                   "SVM", "KNN", "LogReg", "MLP"]
if XGBOOST_OK:  nombres_modelos.append("XGBoost")
if LIGHTGBM_OK: nombres_modelos.append("LightGBM")

todos_res = {}
for nombre in nombres_modelos:
    print(f"▶ Entrenando: {nombre}...")
    try:
        todos_res[nombre] = entrenar_multioutput(
            X_train, X_test, Y_train, Y_test,
            nombre_modelo=nombre, targets=TARGETS,
            encoders=encoders, feat_names=FEAT_COLS,
        )
        r = todos_res[nombre]
        print(f"   ✅ acc_mean={r['acc_mean']:.3f}  acc_subset={r['acc_subset']:.3f}")
    except Exception as e:
        print(f"   ❌ ERROR: {e}")

# ─── Tabla resumen ───────────────────────────────────────────────────────────
df_acc = pd.DataFrame({n: todos_res[n]['acc_per_target'] for n in todos_res}).T
df_acc['mean']       = df_acc.mean(axis=1)
df_acc['std']        = df_acc[TARGETS].std(axis=1)
df_acc['acc_subset'] = pd.Series({n: todos_res[n]['acc_subset'] for n in todos_res})
df_acc = df_acc.sort_values('mean', ascending=False)

print("\n── Resumen final (ordenado por accuracy media descendente) ──")
print(df_acc.round(3).to_string())

## Bloque 7 — Barplot: accuracy media ± std por modelo

In [ ]:
# ─── Barplot horizontal — mejor arriba ───────────────────────────────────────

# ⭐ NUEVO: calcular el baseline "random" específico al problema multi-output.
#    La accuracy media por azar = media de (1/n_clases) sobre los 6 targets.
random_per_target = {t: 1.0 / len(encoders[t].classes_) for t in TARGETS}
random_mean_acc   = np.mean(list(random_per_target.values()))

print(f"📊 Baseline 'random' por target:")
for t, p in random_per_target.items():
    n_c = len(encoders[t].classes_)
    print(f"   {t:<35} {n_c} clases → 1/{n_c} = {p:.3f}")
print(f"   ─────────────────────────────────")
print(f"   Media baseline:  {random_mean_acc:.3f}\n")

df_sorted = df_acc.sort_values('mean', ascending=True)
n         = len(df_sorted)
bar_cols  = plt.get_cmap('YlGn')(np.linspace(0.35, 0.85, n))
sns.set_style('whitegrid')
plt.rc('axes', edgecolor='k')
fig, ax = plt.subplots(figsize=(7, 6), facecolor=BG_COLOR)
ax.set_yticks([])
bars = ax.barh(
    df_sorted.index, df_sorted['mean'],
    color=bar_cols, edgecolor='white', height=0.6,
    xerr=df_sorted['std'], capsize=5,
    error_kw=dict(elinewidth=2, ecolor='#444', capthick=2)
)
for bar, (_, row) in zip(bars, df_sorted.iterrows()):
    ax.text(row['mean'] + row['std'] + 0.008,
            bar.get_y() + bar.get_height() / 2,
            f"{row['mean']:.3f}",
            va='center', ha='left',
            fontsize=FONT_ANNOT-3, fontweight='bold', color='#264653')
for bar, nombre in zip(bars, df_sorted.index):
    ax.text(0.04, bar.get_y() + bar.get_height() / 2,
            nombre, va='center', ha='left',
            fontsize=FONT_TICK - 2, color='white', fontweight='bold')

ax.set_xlabel('Accuracy (mean ± std across 6 targets)', fontsize=FONT_LABEL)
ax.set_yticks([])
ax.set_xlim(0, max(1.18, df_sorted['mean'].max() + df_sorted['std'].max() + 0.15))
ax.set_title('Multi-Output Model Accuracy — Mean ± Std across all targets',
             fontsize=FONT_TITLE-3, fontweight='bold', pad=14)

# ⭐ NUEVO: línea de media general (sin cambios)
ax.axvline(df_sorted['mean'].mean(), color=ACCENT, ls='--', lw=1.8,
           label=f"Overall mean = {df_sorted['mean'].mean():.3f}")
# ⭐ FIX: random baseline calculado para los 6 targets reales, no 0.5
ax.axvline(random_mean_acc, color='gray', ls=':', lw=1.2,
           label=f"Random baseline = {random_mean_acc:.3f}")
ax.legend(fontsize=FONT_LEGEND-4, frameon=True, loc='lower right')

plt.tight_layout()
fname = f"{carpeta}/accuracy_barplot_mean_std{sufijo}.png"
plt.savefig(fname, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {fname}")

## Bloque 8 — Heatmap: modelos × 6 targets

Ahora con **6 targets** en horizontal (en lugar de los 3 de tu v8). Ordenado por accuracy media descendente.

In [ ]:
# ─── Heatmap — modelos × targets ─────────────────────────────────────────────
df_heat = df_acc[TARGETS].loc[df_acc.index]   # mantener orden de df_acc

fig, ax = plt.subplots(figsize=(11, 7), facecolor=BG_COLOR)
sns.heatmap(df_heat, annot=True, fmt='.3f', cmap='YlGn', vmin=0, vmax=1.0,
    linewidths=0.8, linecolor='white',
    annot_kws={'size': FONT_ANNOT, 'weight': 'bold'},
    cbar_kws={'shrink': 0.8, 'label': 'Accuracy'}, ax=ax)
ax.set_title('Multi-Output Accuracy Heatmap — models × targets',
             fontsize=FONT_TITLE-1, fontweight='bold', pad=14)
ax.set_xlabel(''); ax.set_ylabel('')
ax.set_xticklabels([t.replace('_', '\n') for t in TARGETS],
                   fontsize=FONT_TICK-2, rotation=0)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=FONT_TICK, rotation=0)
plt.tight_layout()
fname = f"{carpeta}/accuracy_heatmap{sufijo}.png"
#plt.savefig(fname, dpi=300, bbox_inches='tight')
plt.show()
#print(f"✓ Saved: {fname}")

## Bloque 9 — Curva ROC promediada sobre los 6 targets

Mismo enfoque que en tu v8: para cada modelo, se promedian las curvas One-vs-Rest de **todas las clases de los 6 targets** y se grafican todas en una sola figura.

**Diferencia clave** respecto a v8: ahora `predict_proba` devuelve una **lista** de 6 arrays (uno por target) en vez de un solo array. Lo iteramos.

In [ ]:
# ─── Calcular AUC promedio por modelo ────────────────────────────────────────
def calcular_auc_promedio(res, targets):
    """Curva ROC One-vs-Rest promediada. Salta clases sin positivos en test."""
    Y_proba_list = res['Y_proba']
    Y_test       = res['Y_test']
    all_fpr_list, all_tpr_list, all_auc_list = [], [], []
    n_skipped = 0
    for i, t in enumerate(targets):
        y_t      = Y_test[:, i]
        y_pr_t   = Y_proba_list[i]
        n_clases = y_pr_t.shape[1]
        y_bin    = label_binarize(y_t, classes=list(range(n_clases)))
        if y_bin.shape[1] == 1:
            y_bin = np.column_stack([1 - y_bin, y_bin])
        for k in range(n_clases):
            y_bin_k = y_bin[:, k]
            n_pos = int(y_bin_k.sum())
            if n_pos == 0 or n_pos == len(y_bin_k):
                n_skipped += 1
                continue
            fpr_k, tpr_k, _ = roc_curve(y_bin_k, y_pr_t[:, k])
            all_fpr_list.append(fpr_k)
            all_tpr_list.append(tpr_k)
            all_auc_list.append(auc(fpr_k, tpr_k))
    if not all_fpr_list:
        return np.linspace(0,1,300), np.linspace(0,1,300), float('nan'), n_skipped
    mean_fpr = np.linspace(0, 1, 300)
    mean_tpr = np.zeros_like(mean_fpr)
    for fpr_i, tpr_i in zip(all_fpr_list, all_tpr_list):
        mean_tpr += np.interp(mean_fpr, fpr_i, tpr_i)
    mean_tpr /= len(all_fpr_list)
    auc_val = float(np.mean(all_auc_list))
    return mean_fpr, mean_tpr, auc_val, n_skipped

roc_data = {n: calcular_auc_promedio(todos_res[n], TARGETS) for n in todos_res}
roc_sorted = sorted(roc_data.items(),
                     key=lambda x: -x[1][2] if not np.isnan(x[1][2]) else float('inf'))
print('AUC promedio por modelo:')
for nombre, (_, _, a, n_skip) in roc_sorted:
    print(f"  {nombre:<22} AUC = {a:.4f}   (clases saltadas: {n_skip})")

In [ ]:
# ─── Plot ROC ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8), facecolor=BG_COLOR)
colors = plt.get_cmap('tab10')(np.linspace(0, 0.9, len(roc_sorted)))
for (nombre, (fpr, tpr, auc_val, _)), color in zip(roc_sorted, colors):
    if np.isnan(auc_val):
        continue
    ax.plot(fpr, tpr, color=color, lw=2.5,
            label=f"{nombre:<18}  AUC = {auc_val:.3f}")
ax.plot([-0.02, 1], [-0.02, 1], 'k--', lw=1.8, label='Random classifier')
ax.set_xlim([-0.02, 1.0]); ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('False Positive Rate (FPR)', fontsize=FONT_LABEL)
ax.set_ylabel('True Positive Rate (TPR)', fontsize=FONT_LABEL)
ax.set_title('ROC Curve — All models', fontsize=FONT_TITLE, fontweight='bold', pad=14)
ax.legend(fontsize=11, loc='lower right', frameon=True, framealpha=0.9,
          prop={'family': 'monospace'})
ax.grid(alpha=0.2)
plt.tight_layout()
fname = f"{carpeta}/roc_all_models{sufijo}.png"
#plt.savefig(fname, dpi=300, bbox_inches='tight')
plt.show()
#print(f"✓ Saved: {fname}")

## Bloque 10 — Elegir el mejor modelo y validar a fondo

Mira el ranking de los bloques anteriores (`df_acc` y AUC) y elige.
Por defecto cojo el de mayor `mean accuracy`, pero puedes cambiarlo manualmente.

In [ ]:
# ⭐ Modelo elegido para validación detallada — opciones disponibles:
#    "AdaBoost", "RandomForest", "GradBoost", "ExtraTrees",
#    "SVM", "KNN", "LogReg", "MLP", "XGBoost", "LightGBM"
MODEL = df_acc.index[0]   # ← el de mayor accuracy media. Cambia manualmente si quieres.

_prefijo = MODEL.lower()
res = todos_res[MODEL]

print(f"🏆 Modelo elegido: {MODEL}")
print(f"   Accuracy media:      {res['acc_mean']:.3f}")
print(f"   Accuracy subset:     {res['acc_subset']:.3f}  (= % de filas con TODOS los 6 targets correctos)")
print(f"\n   Accuracy por target:")
for t, a in res['acc_per_target'].items():
    n_clases = len(encoders[t].classes_)
    print(f"     - {t:<35} {a:.3f}   ({n_clases} clases)")

## Bloque 11 — Confusion matrices por target (6 figuras)

In [ ]:
def plot_confusion_matrix_target(res, target, idx, encoders, carpeta, sufijo, prefijo):
    Y_test = res['Y_test']
    Y_pred = res['Y_pred']
    clases = list(encoders[target].classes_)
    y_t    = Y_test[:, idx]
    y_p    = Y_pred[:, idx]
    acc_t  = accuracy_score(y_t, y_p)

    cm      = confusion_matrix(y_t, y_p, labels=list(range(len(clases))))
    cm_norm = confusion_matrix(y_t, y_p, labels=list(range(len(clases))), normalize='true')

    n = len(clases)
    sz = max(6, n * 1.5)
    fig, ax = plt.subplots(figsize=(sz, sz * 0.85), facecolor=BG_COLOR)
    sns.heatmap(cm_norm, annot=True, fmt='.0%',
                cmap=CMAP_CM, vmin=0, vmax=1,
                xticklabels=clases, yticklabels=clases,
                linewidths=0.8, linecolor='white', square=True,
                annot_kws={'size': FONT_ANNOT + 2, 'weight': 'bold'},
                cbar_kws={'shrink': 0.75, 'label': 'Recall'},
                ax=ax)
    for i in range(n):
        for j in range(n):
            color_txt = 'white' if cm_norm[i, j] > 0.55 else '#444'
            ax.text(j + 0.5, i + 0.72, f"n={cm[i,j]}",
                    ha='center', va='center',
                    fontsize=FONT_ANNOT - 1, color=color_txt)
    ax.set_title(
        f"Confusion Matrix · {res['nombre_modelo']} · «{target}»\n"
        f"(normalized by true class — Acc {acc_t:.1%})",
        fontsize=FONT_TITLE-2, fontweight='bold', pad=14
    )
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_xticklabels(clases, rotation=45, ha='right',
                       rotation_mode='anchor', fontsize=FONT_TICK)
    ax.set_yticklabels(clases, rotation=0, fontsize=FONT_TICK)
    plt.tight_layout()
    fname = f"{carpeta}/{prefijo}_confusion_matrix_{target}{sufijo}.png"
    #plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.show()
    #print(f"✓ Saved: {fname}")

for i, t in enumerate(TARGETS):
    plot_confusion_matrix_target(res, t, i, encoders, carpeta, sufijo, _prefijo)

## Bloque 12 — Resumen agregado: Correcto vs Incorrecto por target

Una **sola figura** que resume el rendimiento del modelo sobre los 6 targets a la vez.
Para cada target → tasa de aciertos vs errores en el test set.

Es la vista que mejor responde a *'\¿qué tan bien predice mi modelo en conjunto?'*.

In [ ]:
# ─── Resumen agregado: barras horizontales con accuracy por target ──────────
acc_per   = res['acc_per_target']
acc_df    = pd.DataFrame({'target': list(acc_per.keys()),
                          'acc'   : list(acc_per.values())}).sort_values('acc')
n_total   = len(res['Y_test'])
n_correct = (res['Y_pred'] == res['Y_test']).sum(axis=0)
n_wrong   = n_total - n_correct

fig, ax = plt.subplots(figsize=(10, 6), facecolor=BG_COLOR)
y_pos = np.arange(len(TARGETS))

# Reordenar para que el peor quede abajo (más visible)
order = np.argsort([acc_per[t] for t in TARGETS])
labels_ord = [TARGETS[i] for i in order]
correct_ord = [n_correct[i] for i in order]
wrong_ord   = [n_wrong[i]   for i in order]
acc_ord     = [acc_per[TARGETS[i]] for i in order]

ax.barh(y_pos, correct_ord, color='#2a9d8f', edgecolor='white',
        height=0.65, label='Correct')
ax.barh(y_pos, wrong_ord, left=correct_ord, color='#e63946',
        edgecolor='white', height=0.65, label='Wrong')
for i, (a, c, w) in enumerate(zip(acc_ord, correct_ord, wrong_ord)):
    ax.text(c / 2, i, f"{c}", va='center', ha='center',
            color='white', fontweight='bold', fontsize=FONT_ANNOT)
    ax.text(c + w / 2, i, f"{w}", va='center', ha='center',
            color='white', fontweight='bold', fontsize=FONT_ANNOT)
    ax.text(n_total + n_total * 0.01, i, f"Acc = {a:.1%}",
            va='center', ha='left', fontsize=FONT_TICK,
            color='#264653', fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(labels_ord, fontsize=FONT_TICK)
ax.set_xlabel(f'Test samples (total n = {n_total})', fontsize=FONT_LABEL)
ax.set_title(f"Aggregated validation · {MODEL} · Correct vs Wrong per target\n"
             f"(Subset accuracy = {res['acc_subset']:.1%} of test rows have ALL 6 targets right)",
             fontsize=FONT_TITLE-2, fontweight='bold', pad=14)
ax.set_xlim(0, n_total * 1.18)
ax.legend(loc='lower right', fontsize=FONT_LEGEND, frameon=True)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fname = f"{carpeta}/{_prefijo}_aggregated_correct_vs_wrong{sufijo}.png"
#plt.savefig(fname, dpi=300, bbox_inches='tight')
plt.show()
#print(f"✓ Saved: {fname}")

## Bloque 13 — Top features por target (6 figuras, Top 15)

In [ ]:
def plot_feature_importance_target(res, target, carpeta, sufijo, prefijo, top_n=15):
    feat_imp = res['feat_imp_per_target'][target]
    top      = feat_imp.nlargest(top_n)
    nombre_m = res['nombre_modelo']
    colors   = sns.color_palette('flare', top_n)[::-1]

    fig, ax = plt.subplots(figsize=(10, 7), facecolor=BG_COLOR)
    bars = ax.barh(range(top_n), top.values, color=colors,
                    edgecolor='white', height=0.7)
    for bar, val in zip(bars, top.values):
        ax.text(val + top.values.max() * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va='center',
                fontsize=FONT_ANNOT, color='#264653')
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top.index, fontsize=FONT_TICK)
    ax.invert_yaxis()
    ax.set_xlabel('Importance', fontsize=FONT_LABEL)
    ax.set_title(f"Top {top_n} Feature Importances · {nombre_m} · «{target}»",
                 fontsize=FONT_TITLE-2, fontweight='bold', pad=14)
    ax.axvline(top.values.mean(), color=ACCENT, ls='--', lw=2,
               label=f"Mean = {top.values.mean():.4f}")
    ax.legend(fontsize=FONT_LEGEND, frameon=True,
              loc='lower right', framealpha=0.9)
    plt.tight_layout()
    fname = f"{carpeta}/{prefijo}_top{top_n}_features_{target}{sufijo}.png"
    #plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.show()
    #print(f"✓ Saved: {fname}")

for t in TARGETS:
    plot_feature_importance_target(res, t, carpeta, sufijo, _prefijo, top_n=15)

## Bloque 14 — Top features agregadas (promedio sobre los 6 targets)

**Pregunta que responde**: ¿qué features son universalmente importantes para predecir todo el conjunto de targets?
Se calcula la media de importance de cada feature **sobre los 6 targets** y se muestra el Top 15.

In [ ]:
feat_imp_mean = res['feat_imp_mean']
top_n   = 15
top_agg = feat_imp_mean.nlargest(top_n)

colors = sns.color_palette('crest', top_n)[::-1]
fig, ax = plt.subplots(figsize=(11, 7), facecolor=BG_COLOR)
bars = ax.barh(range(top_n), top_agg.values, color=colors,
                edgecolor='white', height=0.7)
for bar, val in zip(bars, top_agg.values):
    ax.text(val + top_agg.values.max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va='center',
            fontsize=FONT_ANNOT, color='#264653')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_agg.index, fontsize=FONT_TICK)
ax.invert_yaxis()
ax.set_xlabel('Mean importance across 6 targets', fontsize=FONT_LABEL)
ax.set_title(f"Top {top_n} Features — Aggregated across all 6 targets · {MODEL}",
             fontsize=FONT_TITLE-2, fontweight='bold', pad=14)
ax.axvline(top_agg.values.mean(), color=ACCENT, ls='--', lw=2,
           label=f"Mean of top {top_n} = {top_agg.values.mean():.4f}")
ax.legend(fontsize=FONT_LEGEND, frameon=True,
          loc='lower right', framealpha=0.9)
plt.tight_layout()
fname = f"{carpeta}/{_prefijo}_top{top_n}_features_AGGREGATED{sufijo}.png"
#plt.savefig(fname, dpi=300, bbox_inches='tight')
plt.show()
#print(f"✓ Saved: {fname}")

## Bloque 15 — Validation report detallado (precision / recall / F1) por target

`classification_report` de sklearn para cada target — métricas estándar de la literatura:
- **precision** (= de los predichos como clase X, qué % es correcto)
- **recall**    (= de los reales clase X, qué % se ha identificado)
- **f1-score**  (= media armónica de las dos)
- **support**   (= n de samples reales de cada clase en el test set)

In [ ]:
print(f"=== Validation Report · {MODEL} ===\n")
for i, t in enumerate(TARGETS):
    print(f"── {t} ──────────────────────────────────────────")
    # ⭐ labels=range(n_clases) garantiza que TODAS las clases aparecen en el report,
    #    incluso las que falten en el test set (saldrán con support=0).
    #    Sin esto, classification_report falla si target_names tiene más clases
    #    que las realmente presentes en y_test/y_pred.
    print(classification_report(
        res['Y_test'][:, i], res['Y_pred'][:, i],
        labels=list(range(len(encoders[t].classes_))),
        target_names=encoders[t].classes_,
        digits=3, zero_division=0
    ))
print("\n── Multi-Output summary ──")
print(f"  Accuracy media:    {res['acc_mean']:.3f}")
print(f"  Accuracy subset:   {res['acc_subset']:.3f}  (% filas con TODOS los targets correctos)")
# ⭐ hamming_loss de sklearn tampoco soporta multi-class multi-output;
#    se calcula manualmente como % de pares (sample, target) erróneos.
hl = float((res['Y_pred'] != res['Y_test']).mean())
print(f"  Hamming loss:      {hl:.3f}  (% pares (sample, target) erróneos)")

# ────────────────────────────────────────────────────────────────
# 🚀 PARTE 2 — Mejoras y explicación con SHAP
# ────────────────────────────────────────────────────────────────

## Bloque 16 — Feature Selection con `SelectKBest`

**Problema:** la ratio muestras:features es ~150:71 ≈ 2:1, por debajo de lo recomendable.

**Solución:** reducir a las features más informativas usando *mutual information* — mide cuánta información comparte cada feature con cada target. Como tenemos 6 targets, calculamos MI para cada uno y promediamos.

Pasamos de **71 → 25 features**. Esto debería:
- Reducir el overfitting (menos parámetros redundantes que aprender)
- Hacer SHAP más interpretable (menos features = ranking más claro)
- Acelerar el entrenamiento

In [ ]:
# ─── Mutual information per-target ──────────────────────────────────────────
K_FEATURES = 25   # número de features a quedarse

mi_scores = pd.DataFrame(index=FEAT_COLS, columns=TARGETS, dtype=float)
for i, t in enumerate(TARGETS):
    mi_scores[t] = mutual_info_classif(X, Y[:, i], random_state=42)

# Score agregado: media sobre los 6 targets
mi_scores['mean']     = mi_scores[TARGETS].mean(axis=1)
mi_scores_sorted      = mi_scores.sort_values('mean', ascending=False)

# Top K features
top_features          = mi_scores_sorted.head(K_FEATURES).index.tolist()
mask_sel              = np.array([c in top_features for c in FEAT_COLS])
X_sel                 = X[:, mask_sel]
FEAT_COLS_SEL         = [c for c in FEAT_COLS if c in top_features]

print(f"✅ Features seleccionadas: {len(FEAT_COLS_SEL)} de {len(FEAT_COLS)}")
print(f"✅ X_sel shape: {X_sel.shape}\n")
print(f"Top {K_FEATURES} features (mean MI score):")
print(mi_scores_sorted[['mean']].head(K_FEATURES).round(4).to_string())

In [ ]:
# ─── Visualización: scores MI por target ───────────────────────────────────── TARDA MUCHO N EJECUTAR !!!!!!!!!!!!!
fig, ax = plt.subplots(figsize=(10, 8), facecolor=BG_COLOR)
top_show = mi_scores_sorted.head(K_FEATURES)[TARGETS]
sns.heatmap(top_show, annot=True, fmt='.3f', cmap='YlOrRd',
            cbar_kws={'label': 'Mutual Information'}, ax=ax,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 9})
ax.set_title(f'Top {K_FEATURES} Features × Targets — Mutual Information',
             fontsize=FONT_TITLE-2, fontweight='bold', pad=14)
ax.set_xticklabels([t.replace('_', '\n') for t in TARGETS],
                   rotation=0, fontsize=FONT_TICK-2)
plt.tight_layout()
fname = f"{carpeta}/feature_selection_heatmap{sufijo}.png"
#plt.savefig(fname, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {fname}")

## Bloque 17 — Cross-Validation 5-fold con features seleccionadas

Con 30 muestras de test (single split) las métricas son ruidosas — cambiar `random_state=42` a `=43` puede mover el accuracy un 5-10%. **K-fold CV resuelve esto** evaluando el modelo en 5 splits distintos y promediando.

Usamos `KFold` simple (no `StratifiedKFold` porque sklearn no soporta stratify con Y 2D). Cada fold: 120 train / 30 test, igual que antes, pero ahora **medido 5 veces** → media ± std mucho más fiable.

Comparamos: **`X` original (71 features, single split)** vs **`X_sel` reducido (25 features, 5-fold CV)**.

In [ ]:
# ─── 5-fold CV con X_sel ────────────────────────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for nombre in nombres_modelos:
    print(f"▶ {nombre}", end=' ')
    fold_accs = []
    try:
        for fold_train, fold_test in kf.split(X_sel):
            base   = _build_modelo(nombre)              # nueva instancia por fold
            modelo = MultiOutputClassifier(base, n_jobs=-1)
            modelo.fit(X_sel[fold_train], Y[fold_train])
            Y_pred = modelo.predict(X_sel[fold_test])
            acc_per_fold = np.mean([
                accuracy_score(Y[fold_test, i], Y_pred[:, i])
                for i in range(len(TARGETS))
            ])
            fold_accs.append(acc_per_fold)
        cv_results[nombre] = (np.mean(fold_accs), np.std(fold_accs))
        print(f" → {cv_results[nombre][0]:.3f} ± {cv_results[nombre][1]:.3f}")
    except Exception as e:
        print(f" ❌ {e}")

# ─── Tabla comparativa ───────────────────────────────────────────────────────
comp = pd.DataFrame({
    'X original (71f, 1 split)' : df_acc['mean'],
    'X_sel (25f, 5-fold mean)'  : pd.Series({k: v[0] for k, v in cv_results.items()}),
    'X_sel (25f, 5-fold std)'   : pd.Series({k: v[1] for k, v in cv_results.items()}),
})
comp['delta']  = comp['X_sel (25f, 5-fold mean)'] - comp['X original (71f, 1 split)']
comp           = comp.sort_values('X_sel (25f, 5-fold mean)', ascending=False)

print("\n══ Comparación: original vs feature-selected + CV ══")
print(comp.round(3).to_string())

In [ ]:
# ─── Visualización de la comparación ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6), facecolor=BG_COLOR)
y_pos = np.arange(len(comp))
width = 0.4

ax.barh(y_pos - width/2, comp['X original (71f, 1 split)'],
        height=width, color='#999', edgecolor='white',
        label='X original (71 features, 1 split)')
ax.barh(y_pos + width/2, comp['X_sel (25f, 5-fold mean)'],
        height=width, color='#2a9d8f', edgecolor='white',
        xerr=comp['X_sel (25f, 5-fold std)'],
        error_kw=dict(elinewidth=1.5, ecolor='#222'),
        label=f'X_sel ({K_FEATURES} features, 5-fold CV)')

ax.set_yticks(y_pos)
ax.set_yticklabels(comp.index, fontsize=FONT_TICK)
ax.set_xlabel('Accuracy media', fontsize=FONT_LABEL)
ax.set_title('Original vs Feature-Selected + CV',
             fontsize=FONT_TITLE-1, fontweight='bold', pad=14)
ax.legend(loc='lower right', fontsize=FONT_LEGEND)
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(0, 1.05)
plt.tight_layout()
fname = f"{carpeta}/comparacion_original_vs_seleccionado{sufijo}.png"
plt.savefig(fname, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {fname}")

## Bloque 18 — Introducción a SHAP

**SHAP** (*SHapley Additive exPlanations*) es un método para explicar predicciones de **cualquier** modelo de ML. Está basado en los **valores de Shapley** de la teoría de juegos cooperativa, propuestos por Lloyd Shapley (Nobel de Economía 1972).

### La idea

Imagina una predicción como una *ganancia* producida por todas las features actuando en conjunto. SHAP responde:
> *¿Cuánto contribuye cada feature individualmente a esa ganancia?*

Para una predicción `f(x)` cualquiera:

$$f(x) = \\phi_0 + \\sum_{i=1}^{n} \\phi_i$$

Donde:
- `φ₀` (`base_value`) = predicción media del modelo sobre todo el dataset (= lo que el modelo predeciría sin información)
- `φᵢ` (`shap_value`) = cuánto **aleja** la feature `i` la predicción de ese valor base

Los valores SHAP pueden ser **positivos** (la feature empuja la predicción hacia arriba) o **negativos** (la empuja hacia abajo).

### Por qué SHAP y no otros métodos

SHAP es el **único** método de atribución de features que cumple a la vez estas 4 propiedades matemáticas:

| Propiedad | Significado |
|---|---|
| **Eficiencia** | Los SHAP de todas las features suman exactamente la diferencia entre la predicción y el valor base — no se pierde ni se inventa información |
| **Simetría** | Si dos features contribuyen lo mismo, tienen el mismo SHAP |
| **Nulidad** | Una feature que nunca afecta la predicción tiene SHAP = 0 |
| **Aditividad** | Para ensembles (RF, GBM), los SHAP del ensemble = suma de SHAP de los componentes |

Métodos como LIME, gradient×input o permutation importance no cumplen alguna de estas. Por eso SHAP se ha convertido en el estándar.

### Qué te dice cada gráfico

- **Bar plot (resumen)**: ranking global de features por su impacto **medio absoluto** sobre las predicciones. Lo más parecido a *feature importance* tradicional pero con base teórica sólida. *Pregunta que responde: ¿qué features importan en general?*
- **Beeswarm plot**: cada punto = una predicción. Eje X = valor SHAP (cuánto empuja la predicción), Color = valor de la feature (rojo=alto, azul=bajo). *Pregunta que responde: ¿cuándo una feature alta empuja para arriba y cuándo para abajo?*
- **Dependence plot**: cómo varía el SHAP de UNA feature según su valor — captura interacciones no lineales.
- **Force plot**: explica UNA predicción individual — qué features y cuánto la empujaron a su valor.

### Para multi-output con multi-class

Cada uno de nuestros 6 sub-modelos predice una variable de 2-5 clases. `TreeExplainer` devuelve SHAP values con forma:
- `(n_samples, n_features, n_classes)` si multi-class
- `(n_samples, n_features)` si binario

Para visualizar **una sola figura por target** (no una por cada clase), agregamos: `mean(|SHAP|, axis=classes)`. Así obtenemos un ranking de features 'útiles para predecir este target en general'.

### Referencias

- Paper original (Lundberg & Lee, 2017): https://doi.org/10.1038/s42256-019-0138-9
- Librería oficial: https://github.com/shap/shap

## Bloque 19 — Helper SHAP + análisis para el MEJOR modelo

Definimos un helper reutilizable y lo aplicamos al modelo con mejor CV accuracy.

In [ ]:
# ─── Helper: análisis SHAP para UN sub-estimador (un target) ────────────────
def shap_one_target(estimator, X_data, target_name, n_classes,
                     feat_names, plot_dir, prefix, max_display=15):
    """Calcula y plotea SHAP para UN target (un sub-estimador del MultiOutput).

    Devuelve mean |SHAP| por feature (útil para luego comparar entre modelos).
    """
    explainer = shap.TreeExplainer(estimator)
    sv        = explainer(X_data, check_additivity=False)

    # Estructura de sv.values:
    #   (n_samples, n_features)              → binario
    #   (n_samples, n_features, n_classes)   → multi-class
    if sv.values.ndim == 3:
        # Multi-class: para el bar plot agregamos por mean |SHAP| over classes
        shap_bar_data = np.abs(sv.values).mean(axis=2)
        # Para beeswarm usamos la última clase (arbitrario pero consistente)
        shap_bee_data = sv.values[:, :, -1]
        bee_class     = '(última clase)'
    else:
        shap_bar_data = np.abs(sv.values)
        shap_bee_data = sv.values
        bee_class     = '(binario)'

    # ── Bar plot ─────────────────────────────────────────────────────────────
    plt.figure(figsize=(9, 7))
    shap.summary_plot(shap_bar_data, X_data, feature_names=feat_names,
                       plot_type='bar', show=False, max_display=max_display)
    plt.title(f'SHAP Bar · {target_name}\n(mean |SHAP| across {n_classes} clases)',
              fontsize=12, fontweight='bold', pad=10)
    plt.tight_layout()
    fname = f"{plot_dir}/{prefix}_shap_bar_{target_name}.png"
    #plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    # ── Beeswarm ─────────────────────────────────────────────────────────────
    plt.figure(figsize=(9, 7))
    shap.summary_plot(shap_bee_data, X_data, feature_names=feat_names,
                       show=False, max_display=max_display)
    plt.title(f'SHAP Beeswarm · {target_name} {bee_class}',
              fontsize=12, fontweight='bold', pad=10)
    plt.tight_layout()
    fname = f"{plot_dir}/{prefix}_shap_bee_{target_name}.png"
    #plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    return pd.Series(shap_bar_data.mean(axis=0), index=feat_names, name=target_name)

print('✓ Helper SHAP listo')

In [ ]:
# ─── Identificar el MEJOR modelo según CV ───────────────────────────────────
best_model_cv = max(cv_results, key=lambda k: cv_results[k][0])
print(f"🏆 Mejor modelo según CV: {best_model_cv} "
      f"(acc {cv_results[best_model_cv][0]:.3f} ± {cv_results[best_model_cv][1]:.3f})")

# Modelos tree-based aceptan TreeExplainer (rápido)
TREE_BASED = {'RandomForest', 'ExtraTrees', 'GradBoost', 'XGBoost',
              'LightGBM', 'AdaBoost'}

if best_model_cv not in TREE_BASED:
    print(f"\n⚠️  {best_model_cv} no es tree-based — TreeExplainer no aplica.")
    print(f"   Se hace SHAP solo para RandomForest y LightGBM en los siguientes bloques.")
    DO_SHAP_BEST = False
else:
    DO_SHAP_BEST = True

In [ ]:
# ─── Entrenar el mejor modelo en X_sel y aplicar SHAP ───────────────────────
if DO_SHAP_BEST and SHAP_OK:
    # Re-split con features seleccionadas para tener X_test consistente
    X_train_sel, X_test_sel, Y_train, Y_test = train_test_split(
        X_sel, Y, test_size=0.2, random_state=42)

    base_best = _build_modelo(best_model_cv)
    mom_best  = MultiOutputClassifier(base_best, n_jobs=-1)
    mom_best.fit(X_train_sel, Y_train)

    print(f"=== SHAP analysis · {best_model_cv} (modelo ganador) ===\n")
    shap_best = {}
    for i, t in enumerate(TARGETS):
        print(f"▶ Target: {t}")
        shap_best[t] = shap_one_target(
            mom_best.estimators_[i], X_test_sel, t,
            n_classes=len(encoders[t].classes_),
            feat_names=FEAT_COLS_SEL,
            plot_dir=carpeta, prefix=f'{best_model_cv.lower()}_best',
        )
    print(f"\n✅ SHAP analysis para {best_model_cv} completado")
else:
    print("⏭ Saltando SHAP del mejor modelo (no es tree-based o SHAP no instalado)")

## Bloque 20 — SHAP para `RandomForest`

Análisis SHAP independiente sobre RandomForest, entrenado sobre `X_sel` (25 features).
Útil para ver si las features que RF considera importantes coinciden con las del mejor modelo.

In [ ]:
if SHAP_OK:
    base_rf = _build_modelo('RandomForest')
    mom_rf  = MultiOutputClassifier(base_rf, n_jobs=-1)
    mom_rf.fit(X_train_sel, Y_train)

    print("=== SHAP analysis · RandomForest ===\n")
    shap_rf = {}
    for i, t in enumerate(TARGETS):
        print(f"▶ Target: {t}")
        shap_rf[t] = shap_one_target(
            mom_rf.estimators_[i], X_test_sel, t,
            n_classes=len(encoders[t].classes_),
            feat_names=FEAT_COLS_SEL,
            plot_dir=carpeta, prefix='rf',
        )
    print("\n✅ SHAP analysis RandomForest completado")

## Bloque 21 — SHAP para `LightGBM`

In [ ]:
if SHAP_OK and LIGHTGBM_OK:
    base_lgbm = _build_modelo('LightGBM')
    mom_lgbm  = MultiOutputClassifier(base_lgbm, n_jobs=-1)
    mom_lgbm.fit(X_train_sel, Y_train)

    print("=== SHAP analysis · LightGBM ===\n")
    shap_lgbm = {}
    for i, t in enumerate(TARGETS):
        print(f"▶ Target: {t}")
        shap_lgbm[t] = shap_one_target(
            mom_lgbm.estimators_[i], X_test_sel, t,
            n_classes=len(encoders[t].classes_),
            feat_names=FEAT_COLS_SEL,
            plot_dir=carpeta, prefix='lgbm',
        )
    print("\n✅ SHAP analysis LightGBM completado")

## Bloque 22 — Comparación: Top features según SHAP en los 3 modelos

**Pregunta interesante para el TFG**: ¿los 3 modelos (mejor, RF, LGBM) están de acuerdo en qué features importan?

- Si **coinciden mucho** → tu modelo ha encontrado una señal robusta. Esas features son biológicamente relevantes.
- Si **discrepan** → cada modelo se apoya en señales distintas. Indica que la relación features→target es compleja y depende del tipo de modelo.

In [ ]:
# ─── Construir matriz comparativa de top features ───────────────────────────
TOP_N = 15

# Para cada modelo, promediar mean |SHAP| sobre los 6 targets
shap_summary = {}
if DO_SHAP_BEST and SHAP_OK:
    shap_summary[best_model_cv] = pd.concat(shap_best.values(), axis=1).mean(axis=1)
if SHAP_OK:
    shap_summary['RandomForest'] = pd.concat(shap_rf.values(), axis=1).mean(axis=1)
    if LIGHTGBM_OK:
        shap_summary['LightGBM'] = pd.concat(shap_lgbm.values(), axis=1).mean(axis=1)

df_shap_summary = pd.DataFrame(shap_summary)
df_shap_summary = df_shap_summary.sort_values(
    df_shap_summary.columns[0], ascending=False).head(TOP_N)

print(f"Top {TOP_N} features globales según SHAP (mean across 6 targets):\n")
print(df_shap_summary.round(4).to_string())

In [ ]:
# ─── Heatmap comparativo ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8), facecolor=BG_COLOR)
sns.heatmap(df_shap_summary, annot=True, fmt='.3f',
            cmap='YlGnBu', cbar_kws={'label': 'mean |SHAP|'},
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 10}, ax=ax)
ax.set_title(f'Top {TOP_N} Features — SHAP comparado entre modelos',
             fontsize=FONT_TITLE-2, fontweight='bold', pad=14)
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout()
fname = f"{carpeta}/shap_comparison_top_features{sufijo}.png"
#plt.savefig(fname, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {fname}")

# ────────────────────────────────────────────────────────────────
# 🚀 PARTE 3 — Datos para DISCUSION 
# ────────────────────────────────────────────────────────────────


## Bloque 23 — Complejidad combinatoria


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOQUE 23 — Complejidad combinatoria del problema multi-output
# ═══════════════════════════════════════════════════════════════════════════
# Justifica por qué subset_accuracy = 3.3% NO es bajo: el espacio de salida
# crece multiplicativamente con el número de targets y de clases por target.

from functools import reduce
from operator import mul

# ─── Número de clases por target ────────────────────────────────────────────
n_classes_per_target = {t: len(encoders[t].classes_) for t in TARGETS}

print("📊 Espacio de salida por target:\n")
for t in TARGETS:
    n = n_classes_per_target[t]
    classes = list(encoders[t].classes_)
    print(f"  {t:<35} → {n} clases  {classes}")

# ─── Escenarios: 6, 3 mejores, 2 mejores, 1 mejor ───────────────────────────
acc_per = res['acc_per_target']
sorted_by_acc = sorted(acc_per, key=acc_per.get, reverse=True)

scenarios = [
    ('6 targets (caso actual)',         TARGETS),
    ('3 targets más predecibles',       sorted_by_acc[:3]),
    ('2 targets más predecibles',       sorted_by_acc[:2]),
    ('1 target más predecible (MCc)',   sorted_by_acc[:1]),
]

print("\n" + "═"*80)
print("ANÁLISIS COMBINATORIO:")
print("═"*80)

results_table = []
for label, subset in scenarios:
    n_combos = reduce(mul, [n_classes_per_target[t] for t in subset])
    p_random = 1.0 / n_combos
    p_indep  = reduce(mul, [acc_per[t] for t in subset])     # asumiendo independencia
    p_real   = res['acc_subset'] if len(subset) == 6 else None

    print(f"\n● {label}")
    print(f"  Targets             : {subset}")
    print(f"  Combinaciones (n)   : {n_combos:,}")
    print(f"  Acc. esperada azar  : {100*p_random:>7.3f}%   (1 / n_combinaciones)")
    print(f"  Acc. esperada indep.: {100*p_indep :>7.2f}%   (∏ accuracy_individual)")
    if p_real is not None:
        improvement = p_real / p_random
        print(f"  Acc. REAL del modelo: {100*p_real  :>7.2f}%   "
              f"(= {improvement:.0f}× mejor que el azar)")

    results_table.append({
        'escenario': label, 'n_targets': len(subset),
        'n_combinaciones': n_combos,
        'acc_azar_%': 100*p_random,
        'acc_indep_%': 100*p_indep,
    })

df_combos = pd.DataFrame(results_table)
print("\n" + "═"*80)
print("TABLA RESUMEN:")
print("═"*80)
print(df_combos.round(3).to_string(index=False))

# ─── Figura: crecimiento combinatorio ──────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), facecolor=BG_COLOR)

# Subplot 1: combinaciones (log)
xpos = list(range(len(scenarios)))
combos = [s['n_combinaciones'] for s in results_table]
colors_bar = ['#e76f51', '#f4a261', '#e9c46a', '#2a9d8f']
bars1 = ax1.bar(xpos, combos, color=colors_bar, edgecolor='white', width=0.6)
ax1.set_yscale('log')
for bar, val in zip(bars1, combos):
    ax1.text(bar.get_x()+bar.get_width()/2, val*1.4, f'{val:,}',
             ha='center', fontsize=FONT_ANNOT, fontweight='bold')
ax1.set_xticks(xpos)
ax1.set_xticklabels([f'{s["n_targets"]} target{"s" if s["n_targets"]>1 else ""}'
                     for s in results_table])
ax1.set_ylabel('Combinaciones posibles (escala log)', fontsize=FONT_LABEL)
ax1.set_title('Crecimiento del espacio de salida',
              fontsize=FONT_TITLE-2, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Subplot 2: accuracy esperada azar vs independencia
azar = [s['acc_azar_%'] for s in results_table]
indep = [s['acc_indep_%'] for s in results_table]
w = 0.35
ax2.bar([x-w/2 for x in xpos], azar, width=w, color='#bbb', label='Azar puro',
        edgecolor='white')
ax2.bar([x+w/2 for x in xpos], indep, width=w, color='#2a9d8f', label='Por independencia',
        edgecolor='white')
for x, (a, i) in enumerate(zip(azar, indep)):
    ax2.text(x-w/2, a+1, f'{a:.2f}%', ha='center', fontsize=10)
    ax2.text(x+w/2, i+1, f'{i:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax2.set_xticks(xpos)
ax2.set_xticklabels([f'{s["n_targets"]} target{"s" if s["n_targets"]>1 else ""}'
                     for s in results_table])
ax2.set_ylabel('Subset accuracy esperada (%)', fontsize=FONT_LABEL)
ax2.set_title('Subset accuracy vs número de targets',
              fontsize=FONT_TITLE-2, fontweight='bold')
ax2.legend(fontsize=FONT_LEGEND-1, loc='upper left')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
fname = f"{carpeta}/combinaciones_vs_targets{sufijo}.png"
plt.savefig(fname, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {fname}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOQUE 23.2 — Análisis combinatorio · versión visual refinada
# ═══════════════════════════════════════════════════════════════════════════
# Requiere `results_table` calculado en Bloque 23. Misma data, diseño nuevo.

import matplotlib as mpl
import matplotlib.ticker as ticker

# ── Paleta scientific custom ────────────────────────────────────────────────
PAL_PRIMARY  = '#1A365D'   # azul profundo
PAL_ACCENT   = '#2D8B7A'   # teal
PAL_WARM     = '#E07856'   # coral
PAL_GRAY     = '#404040'
PAL_BG_LIGHT = 'white'

# ── Sanity check ────────────────────────────────────────────────────────────
try:
    _ = results_table
except NameError:
    raise NameError("Necesitas haber ejecutado el Bloque 23 antes.")

# ── Styling global temporal ─────────────────────────────────────────────────
old_rc = mpl.rcParams.copy()
plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.edgecolor'   : '#888',
    'axes.linewidth'   : 0.8,
    'xtick.color'      : '#444',
    'ytick.color'      : '#444',
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7), facecolor=PAL_BG_LIGHT)

# ── DATOS — ordenados de menos a más targets para narrativa L→R ─────────────
ordered = sorted(results_table, key=lambda r: r['n_targets'])
nt_arr  = [r['n_targets']        for r in ordered]
combos  = [r['n_combinaciones']  for r in ordered]
azar    = [r['acc_azar_%']       for r in ordered]
indep   = [r['acc_indep_%']      for r in ordered]
labels  = [f'{nt} target{"s" if nt>1 else ""}' for nt in nt_arr]

# ═══════════════════════════════════════════════════════════════════════════
# SUBPLOT 1 — Crecimiento del espacio de salida (gradient bars + boxed labels)
# ═══════════════════════════════════════════════════════════════════════════
xpos   = np.arange(len(combos))
cmap   = mpl.colormaps['YlGnBu']
colors = [cmap(0.25 + 0.55 * i / max(1, len(combos)-1)) for i in range(len(combos))]

bars = ax1.bar(xpos, combos, color=colors, edgecolor='white',
                width=0.55, zorder=3, linewidth=1.5)
ax1.set_yscale('log')

# Etiquetas con caja: número en negrita + borde de color
for x, (bar, val) in enumerate(zip(bars, combos)):
    ax1.text(bar.get_x() + bar.get_width()/2, val * 1.7,
             f'{val:,}',
             ha='center', va='bottom',
             fontsize=12, fontweight='bold', color=PAL_PRIMARY,
             bbox=dict(boxstyle='round,pad=0.45', facecolor='white',
                       edgecolor=colors[x], linewidth=1.8))

ax1.set_xticks(xpos)
ax1.set_xticklabels(labels, fontsize=11, color=PAL_GRAY)
ax1.set_ylabel('Combinaciones posibles', fontsize=12, color=PAL_GRAY,
               fontweight='bold', labelpad=10)
ax1.set_title("Combinaciones posibles del clasificador multi-output",
              fontsize=15, color="black", fontweight='bold',
              pad=25, loc='left')
ax1.text(0, 1.04,
         'Cada target adicional multiplica las combinaciones por su nº de clases',
         transform=ax1.transAxes, fontsize=10, color='#888', style='italic')

# Eje Y: log scale con notación matemática 10ⁿ
ax1.yaxis.set_major_locator(ticker.LogLocator(numticks=8))
ax1.yaxis.set_major_formatter(ticker.LogFormatterMathtext())
ax1.grid(axis='y', alpha=0.25, linestyle='--', zorder=1)
ax1.set_ylim(1, max(combos) * 8)
ax1.tick_params(axis='x', length=0)

# ═══════════════════════════════════════════════════════════════════════════
# SUBPLOT 2 — Slope chart con área sombreada de "ventaja del modelo"
# ═══════════════════════════════════════════════════════════════════════════
# Para que la X sea uniforme visualmente, usamos xpos (categorical) en vez de nt_arr
ax2.fill_between(xpos, azar, indep, alpha=0.18, color=PAL_ACCENT, zorder=1,
                 label='Ventaja del modelo')

# Línea de azar (gris, con marker hueco)
ax2.plot(xpos, azar, 'o-', color='#999', linewidth=2.2,
         markersize=11, markerfacecolor='white', markeredgewidth=2.5,
         label='Azar puro', zorder=3)

# Línea de modelo por independencia (teal sólido)
ax2.plot(xpos, indep, 'o-', color=PAL_ACCENT, linewidth=3,
         markersize=12, markerfacecolor=PAL_ACCENT, markeredgecolor='white',
         markeredgewidth=2, label='Modelo (∏ accuracy)', zorder=4)

# Etiquetas: valores encima (modelo) y debajo (azar)
for x, (a, i) in enumerate(zip(azar, indep)):
    # Modelo (arriba)
    ax2.annotate(f'{i:.1f}%', xy=(x, i), xytext=(0, 14),
                 textcoords='offset points', ha='center',
                 fontsize=11, color=PAL_ACCENT, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                           edgecolor=PAL_ACCENT, linewidth=1.2))
    # Azar (abajo)
    label_a = f'{a:.2f}%' if a < 1 else f'{a:.1f}%'
    ax2.annotate(label_a, xy=(x, a), xytext=(0, -22),
                 textcoords='offset points', ha='center',
                 fontsize=10, color='#666', fontweight='bold')

# Eje Y log para mostrar bien el decay (50% → 0.02% son 3 órdenes)
ax2.set_yscale('log')
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(
    lambda y, _: f'{y:.1f}%' if y >= 1 else f'{y:.2f}%'))

ax2.set_xticks(xpos)
ax2.set_xticklabels(labels, fontsize=11, color=PAL_GRAY)
ax2.set_ylabel('Subset accuracy esperada (escala log)', fontsize=12,
               color=PAL_GRAY, fontweight='bold', labelpad=10)
ax2.set_title('La penalización del multi-output',
              fontsize=15, color="black", fontweight='bold',
              pad=25, loc='left')
ax2.text(0, 1.04,
         'El área verde = ventaja del modelo sobre el azar puro',
         transform=ax2.transAxes, fontsize=10, color='#888', style='italic')

ax2.grid(axis='y', alpha=0.25, linestyle='--')
ax2.set_ylim(min(azar) * 0.3, max(indep) * 3)
ax2.tick_params(axis='x', length=0)

# Leyenda elegante
leg = ax2.legend(fontsize=10, loc='upper right', frameon=True,
                 framealpha=0.95, edgecolor='#ddd', fancybox=True)
leg.get_frame().set_linewidth(0.8)

# ── Suptitle y guardado ─────────────────────────────────────────────────────
fig.suptitle('Complejidad combinatoria del problema multi-output',
             fontsize=17, fontweight='bold', color="black",
             y=1.04, x=0.5)

plt.tight_layout()
fname = f"{carpeta}/combinaciones_vs_targets_2{sufijo}.png"
plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor=PAL_BG_LIGHT)
plt.show()
print(f"✓ Saved: {fname}")

# ── Restaurar matplotlib defaults para no afectar plots posteriores ─────────
mpl.rcParams.update(old_rc)
sns.set_style('whitegrid')

## Bloque 24 — Distribución de clases en todos los targets

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOQUE 24 — Distribución de clases por target (análisis de desbalance)
# ═══════════════════════════════════════════════════════════════════════════
# Identifica clases minoritarias: las que aparecen en <10% de las muestras.
# Estas clases son las que el modelo "no aprende a predecir" porque ve
# muy pocos ejemplos en train.

print("📊 Distribución de clases por target (dataset completo):\n")
print(f"{'Target':<35} {'Clase':<25} {'n':>5} {'%':>7}   Estado")
print("─" * 90)

distr_records = []
for t in TARGETS:
    counts = df[t].value_counts()
    total = counts.sum()
    for cls in encoders[t].classes_:
        n = int(counts.get(cls, 0))
        p = 100*n/total if total > 0 else 0
        status = "⚠️ minoritaria" if p < 10 else ("✓ mayoritaria" if p > 40 else "")
        print(f"{t:<35} {cls:<25} {n:>5} {p:>6.1f}%   {status}")
        distr_records.append({'target': t, 'clase': cls, 'n': n, 'pct': p})
    print()

# ─── Resumen de imbalance por target ──────────────────────────────────────
print("═"*80)
print("RESUMEN DEL DESBALANCE:")
print("═"*80)
for t in TARGETS:
    counts = df[t].value_counts()
    n_minority = (counts/counts.sum() < 0.10).sum()
    imbalance_ratio = counts.max() / max(counts.min(), 1)
    print(f"  {t:<35}  ratio mayor/menor = {imbalance_ratio:>5.1f}×  "
          f"({n_minority} clase{'s' if n_minority != 1 else ''} <10%)")

# ─── Figura: 2×3 grid de histogramas con línea de minoría ──────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9), facecolor=BG_COLOR)
for ax, t in zip(axes.flatten(), TARGETS):
    counts = df[t].value_counts().reindex(encoders[t].classes_, fill_value=0)
    total = counts.sum()
    threshold_10pct = 0.10 * total

    bar_colors = ['#e63946' if v < threshold_10pct else PALETTE[i % len(PALETTE)]
                  for i, v in enumerate(counts.values)]
    bars = ax.bar(range(len(counts)), counts.values,
                   color=bar_colors, edgecolor='white')
    for bar, n in zip(bars, counts.values):
        if n > 0:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(counts)*0.02,
                    f'{n}\n({100*n/total:.0f}%)',
                    ha='center', fontsize=9, fontweight='bold')

    ax.axhline(threshold_10pct, ls='--', color='#e63946', alpha=0.7, lw=1.5)
    ax.text(len(counts)-0.5, threshold_10pct*1.02, '  10%',
            color='#e63946', fontsize=9, va='bottom')

    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(counts.index, rotation=25, ha='right', fontsize=9)
    ax.set_title(f'{t}\n(n={total} pocillos)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Pocillos')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Distribución de clases — barras rojas = clases minoritarias (<10%)',
             fontsize=FONT_TITLE-1, fontweight='bold', y=1.0)
plt.tight_layout()
fname = f"{carpeta}/distribucion_clases_por_target{sufijo}.png"
plt.savefig(fname, dpi=300, bbox_inches='tight')
plt.show()
print(f"\n✓ Saved: {fname}")